# Laboratory Exercise 1: Food Delivery

City Delivery is a delivery service that brings meals from restaurants across Macedonia to the door. Its pages are a stored snapshot in `data/site`, and the setup serves them from this machine, so every request stays on the local loopback address. The orders are synthetic and describe no real restaurant, courier, or customer.

Answer each request in the cells below it, and add as many cells as you need. The laboratory test asks for values and decisions that your notebook computes, so keep every result visible, run the notebook from top to bottom before you submit it, and make sure it runs on a machine without internet access.

## Setup

The cells in this section are written already. Run them before the tasks.

In [3]:
%pip install playwright

   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/38.8 MB ? eta -:--:--
   -- ------------------------------------- 2.1/38.8 MB 7.9 MB/s eta 0:00:05
   --- ------------------------------------ 3.4/38.8 MB 7.3 MB/s eta 0:00:05
   ----- ---------------------------------- 5.0/38.8 MB 7.4 MB/s eta 0:00:05
   ------- -------------------------------- 7.1/38.8 MB 8.0 MB/s eta 0:00:04
   -------- ------------------------------- 8.4/38.8 MB 7.8 MB/s eta 0:00:04
   --------- ------------------------------ 9.2/38.8 MB 7.4 MB/s eta 0:00:05
   --------- ------------------------------ 9.2/38.8 MB 7.4 MB/s eta 0:00:05
   ----------- ---------------------------- 10.7/38.8 MB 6.2 MB/s eta 0:00:05
   ------------ --------------------------- 11.8/38.8 MB 6.2 MB/s eta 0:00:05
   ------------- -------------------------- 13.4/38.8 MB 6.2 MB/s eta 0:00:05
   --------------- ------------------------ 14.7/38.8 MB 6.3 MB/s eta 0:00:04
   ------

In [2]:
import json
import re
import threading
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import parse_qs, urljoin, urlparse

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from bs4 import BeautifulSoup, Tag
from playwright.async_api import async_playwright
from scipy import stats
from statsmodels.stats.multitest import multipletests

DATA = Path("data")
# Every random step draws from this generator, so a rerun gives the same result.
rng = np.random.default_rng(2026)

In [3]:
# The course style.
PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#1e2a24"]
GROUND, INK, GRID = "#ffffff", "#1e2a24", "#e2e6e4"
plt.rcParams.update(
    {
        "figure.figsize": (8, 4.5),
        "figure.facecolor": GROUND,
        "figure.constrained_layout.use": True,
        "savefig.facecolor": GROUND,
        "savefig.transparent": False,
        "font.family": "sans-serif",
        "font.sans-serif": ["Archivo", "DejaVu Sans"],
        "font.size": 12,
        "text.color": INK,
        "axes.facecolor": GROUND,
        "axes.edgecolor": INK,
        "axes.labelcolor": INK,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.axisbelow": True,
        "axes.prop_cycle": plt.cycler(color=PALETTE),
        "grid.color": GRID,
        "xtick.color": INK,
        "ytick.color": INK,
        "legend.frameon": False,
        "image.cmap": "cividis",
    }
)
sns.set_palette(PALETTE)

In [4]:
# Every path is a file under the site directory.
class Service(SimpleHTTPRequestHandler):
    extensions_map = {
        **SimpleHTTPRequestHandler.extensions_map,
        ".html": "text/html; charset=utf-8",
    }

    def log_message(self, format: str, *args: object) -> None:
        pass


# Port 0 lets the system choose a free port, so a second running notebook never collides.
server = ThreadingHTTPServer(("127.0.0.1", 0), partial(Service, directory=DATA / "site"))
threading.Thread(target=server.serve_forever, daemon=True).start()
BASE = f"http://127.0.0.1:{server.server_port}/"
BASE

'http://127.0.0.1:51773/'

## Task 1: Collect the Orders

The service lists its delivered orders across several pages, and the first one is `catalogue/page-1.html` under `BASE`. Collect every order into a table named `orders`, with one row per order and these columns:

- `order_id`, the identifier of the order,
- `restaurant`, `cuisine`, and `city`,
- `distance` and `unit`, the distance from the restaurant to the door and the unit it is quoted in, exactly as the order states them,
- `minutes`, the minutes the delivery took,
- `slot`, `weather`, `payment`, and `rating`, the stars out of 5 that the customer gave, and
- `by_restaurant`, which is true when the restaurant delivered the order itself and false when a courier of the service did.

An order that does not state a field has a missing value in that column. Report how many orders the service lists and over how many pages they are spread.

In [7]:
url = BASE + "catalogue/page-1.html"

response = requests.get(url, timeout=10)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

cards = soup.select("article.order")

print("Status:", response.status_code)
print("Order on page 1:", len(cards))
print("First order ID:", cards[0]["data-id"])


Status: 200
Order on page 1: 12
First order ID: 310069


In [8]:
print(cards[0].prettify())

<article class="card order" data-id="310069" id="order-310069">
 <h2 class="title">
  <a href="#order-310069">
   Salad Bowl
  </a>
 </h2>
 <p class="place">
  <span class="cuisine">
   Salad
  </span>
  ,
  <span class="city">
   Skopje
  </span>
 </p>
 <p class="delivery figure">
  Delivered in 23 minutes.
 </p>
 <p class="summary">
  A trip of 5.0 km from the restaurant to the door.
 </p>
 <ul class="tags">
  <li class="slot">
   Late night
  </li>
  <li class="weather">
   Clear
  </li>
  <li class="payment">
   Cash
  </li>
  <li class="rating">
   Rated 5 of 5
  </li>
 </ul>
 <p class="courier muted">
  Delivered by a City Delivery courier.
 </p>
</article>



In [9]:
card = cards[0]

order_id = int(card["data-id"])
restaurant = card.select_one(".title").get_text(strip=True)
cuisine = card.select_one(".cuisine").get_text(strip=True)
city = card.select_one(".city").get_text(strip=True)

print("Order ID:", order_id)
print("Restaurant:", restaurant)
print("Cuisine:", cuisine)
print("City:", city)

Order ID: 310069
Restaurant: Salad Bowl
Cuisine: Salad
City: Skopje


In [10]:
# Delivery time
delivery = card.select_one(".delivery.figure")

delivery_text = delivery.get_text(strip=True) if delivery else ""
minutes_match = re.search(r"(\d+)\s+minutes", delivery_text)

minutes = int(minutes_match.group(1)) if minutes_match else None

# Distance and unit
summary = card.select_one(".summary")
summary_text = summary.get_text(strip=True) if summary else ""

distance_match = re.search(r"(\d[\d .]*)\s+(km|m)\b", summary_text)

distance = float(distance_match.group(1).replace(" ", "")) if distance_match else None
unit = distance_match.group(2) if distance_match else None

print("Minutes:", minutes)
print("Distance:", distance)
print("Unit:", unit)

Minutes: 23
Distance: 5.0
Unit: km


In [11]:
# Extract slot, weather and payment
slot = card.select_one(".slot").get_text(strip=True)
weather = card.select_one(".weather").get_text(strip=True)
payment = card.select_one(".payment").get_text(strip=True)

# Extract rating
rating_element = card.select_one(".rating")

rating_text = rating_element.get_text(strip=True) if rating_element else ""
rating_match = re.search(r"Rated (\d+) of 5", rating_text)

rating = int(rating_match.group(1)) if rating_match else None

# Check who delivered the order
courier_element = card.select_one(".courier")
courier_text = courier_element.get_text(strip=True) if courier_element else ""

by_restaurant = "by the restaurant" in courier_text.lower() if courier_text else None

print("Slot:", slot)
print("Weather:", weather)
print("Payment:", payment)
print("Rating:", rating)
print("By restaurant:", by_restaurant)

Slot: Late night
Weather: Clear
Payment: Cash
Rating: 5
By restaurant: False


In [12]:
def parse_order(card):

    def optional_text(selector):
        element = card.select_one(selector)
        return element.get_text(strip=True) if element else None

    # Delivery time
    delivery_text = optional_text(".delivery.figure") or ""
    minutes_match = re.search(r"(\d+)\s+minutes", delivery_text)

    minutes = int(minutes_match.group(1)) if minutes_match else None

    # Distance
    summary_text = optional_text(".summary") or ""
    distance_match = re.search(r"(\d[\d .]*)\s+(km|m)\b", summary_text)

    distance = float(distance_match.group(1).replace(" ", "")) if distance_match else None
    unit = distance_match.group(2) if distance_match else None

    # Rating
    rating_text = optional_text(".rating") or ""
    rating_match = re.search(r"Rated\s+(\d+)\s+of\s+5", rating_text)

    rating = int(rating_match.group(1)) if rating_match else None

    # Courier
    courier_text = optional_text(".courier")

    by_restaurant = (
        "by the restaurant" in courier_text.lower()
        if courier_text else None
    )

    return {
        "order_id": int(card["data-id"]),
        "restaurant": optional_text(".title"),
        "cuisine": optional_text(".cuisine"),
        "city": optional_text(".city"),
        "distance": distance,
        "unit": unit,
        "minutes": minutes,
        "slot": optional_text(".slot"),
        "weather": optional_text(".weather"),
        "payment": optional_text(".payment"),
        "rating": rating,
        "by_restaurant": by_restaurant
    }

In [15]:
# Start from the first page
page_url = BASE + "catalogue/page-1.html"

all_orders = []
page_count = 0

while page_url:

    response = requests.get(page_url, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    cards = soup.select("article.order")

    for card in cards:
        order = parse_order(card)
        all_orders.append(order)

    page_count += 1

    next_link = soup.select_one("a.next")

    if next_link:
        page_url = urljoin(page_url, next_link["href"])
    else:
        page_url = None

orders = pd.DataFrame(all_orders)

print("Total pages:", page_count)
print("Total orders:", len(orders))
print("Shape:", orders.shape)

orders.head()

Total pages: 16
Total orders: 192
Shape: (192, 12)


,order_id,restaurant,cuisine,city,distance,unit,minutes,slot,weather,payment,rating,by_restaurant
0,310069,Salad Bowl,Salad,Skopje,5.0,km,23.0,Late night,Clear,Cash,5.0,False
1,283894,Pizza House,Pizza,Skopje,3650.0,m,30.0,Lunch,Clear,Card,3.0,False
2,832866,Sushi Bar,Sushi,Skopje,4700.0,m,NaN,Lunch,Rain,Cash,3.0,True
3,582431,Home Kitchen,Traditional,Skopje,2.7,km,25.0,Lunch,Rain,Card,4.0,False
4,662069,Snack Corner,Small plates,Skopje,2.4,km,26.0,Late night,Clear,Card,4.0,True


In [16]:
print("Total orders:", len(orders))
print("Unique order IDs:", orders["order_id"].nunique())
print("Duplicate IDs:", orders["order_id"].duplicated().sum())

print("\nMissing values:")
print(orders.isna().sum())

Total orders: 192
Unique order IDs: 192
Duplicate IDs: 0

Missing values:
order_id          0
restaurant        0
cuisine           0
city              0
distance          0
unit              0
minutes          42
slot              0
weather           0
payment           0
rating           24
by_restaurant     0
dtype: int64


## Task 2: Add the Receipts

The home page of the service, `index.html` under `BASE`, points to a page of receipts. Collect every receipt the page holds, and add the items, the total, and the tip of each order to `orders`, so that every order appears exactly once. Name the result `df`.

Report how many receipts the page says it holds, how many you collected, and what you did about any receipt that would put an order in `df` more than once.

In [ ]:
# Write your solution here.

## Task 3: Describe the Orders

Answer each request below from `df`. Support each answer with the numbers that establish it, and with a figure wherever the request asks you to show something.

### 3.1 Column Types

Give every column of `df` a type that matches what it measures. The slots have a natural order, from lunch to late night, and the type of `slot` keeps it.

In [ ]:
# Write your solution here.

### 3.2 What Is Missing

Count the missing values in each column, show where they fall, and find out whether the absence of a value depends on another column.

In [ ]:
# Write your solution here.

### 3.3 Distances in Two Units

Some orders quote the distance in metres and others in kilometres. Compare the quoted distances within each unit, then add a column `distance_km` that states every distance in kilometres.

In [ ]:
# Write your solution here.

### 3.4 The Delivery Time

Describe the delivery time by its centre, its spread, and the shape of its distribution, and show the distribution. Decide which summary of the centre describes a typical order better.

In [ ]:
# Write your solution here.

### 3.5 Delivery Time by Slot

Compare the delivery time across the slots.

In [ ]:
# Write your solution here.

### 3.6 Orders by City

Show how the orders are spread across the cities, and give the share of orders in each city.

In [ ]:
# Write your solution here.

### 3.7 Items and Total

Show how the number of items and the total of an order move together, and measure how strongly they do. Decide whether the relationship is close to a straight line.

In [ ]:
# Write your solution here.

### 3.8 Every Pair of Numeric Columns

Show how strongly every pair of numeric columns moves together, measured by the Spearman correlation, and name the pair that moves together most strongly.

In [ ]:
# Write your solution here.

### 3.9 Delivery Time by City and Slot

Find the combination of city and slot with the highest median delivery time, counting only the combinations with at least three orders that state a delivery time.

In [ ]:
# Write your solution here.

## Task 4: Delivery Times

Use a significance level of 5 % and a confidence level of 95 % for every decision and every range in this task and the next. For each question, state the decision and give the range beside it. The questions of this task concern the orders that state a delivery time.

### 4.1 The Advertised Time

The home page states an average delivery time. Decide whether the average delivery time of the orders differs from that statement, and give a range for the average delivery time.

In [ ]:
# Write your solution here.

### 4.2 Rain and Clear Weather

Decide whether the orders delivered in rain and in clear weather differ in their average delivery time, and give a range for the difference, taken as rain minus clear.

In [ ]:
# Write your solution here.

## Task 5: Slots and Payments

The questions of this task concern all orders.

### 5.1 Payment by Slot

Decide whether the payment method depends on the slot of an order, and give a range for the share of cash payments in each slot, using a percentile bootstrap.

In [ ]:
# Write your solution here.

### 5.2 The Mix of Slots

The home page states how the orders divide among the slots. Decide whether the orders match that statement, and give a range for the share of late-night orders, using a percentile bootstrap.

In [ ]:
# Write your solution here.

In [ ]:
server.shutdown()